<a href="https://colab.research.google.com/github/kanavG10/adsrp-parkinson-s/blob/main/02_keypoint_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gait keypoint extraction (MediaPipe BlazePose)

Extracts 33 body keypoints per frame from walking video, renders a skeleton
overlay, and derives gait signals (cadence, arm-swing asymmetry).

This is the **input stage** for the V-JEPA masking study: the keypoint groups
defined in `src/pose_topology.py` are the six pretraining conditions.

**No GPU needed** — BlazePose runs on CPU. `Runtime > Change runtime type` can
stay on CPU; a GPU runtime will not make this faster.


## 1. Setup


In [ ]:
!git clone -q https://github.com/kanavG10/adsrp-parkinson-s.git repo
%cd repo
!pip install -q -r requirements.txt
print('installed')


If pip warns about a dependency conflict and offers **Restart session**, click it,
then re-run from the `%cd repo` cell (the clone survives a restart).

MediaPipe pins `numpy`/`protobuf` versions that sometimes disagree with Colab's
preinstalled stack. The restart is normal and the pipeline works after it.


In [ ]:
import mediapipe as mp, cv2, sys
print('python    ', sys.version.split()[0])
print('mediapipe ', mp.__version__)
print('opencv    ', cv2.__version__)
# Legacy mp.solutions.pose was removed in 0.10.30+; this repo uses the Tasks API.
print('mp.solutions present:', hasattr(mp, 'solutions'))


## 2. Pose model

`full` is the accuracy/speed sweet spot. `heavy` is more accurate on distant
subjects but ~40% slower; `lite` is for quick iteration.


In [ ]:
!mkdir -p models
!wget -q -O models/pose_landmarker_full.task \
  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!ls -lh models/


## 3. Demo video

figshare [10.6084/m9.figshare.19929893](https://doi.org/10.6084/m9.figshare.19929893.v1)
("Walking videos", CC0). The dataset is a single 2.35 GB zip, so `remotezip` pulls
one file out of it over HTTP range requests instead of downloading the archive.

**These are not Parkinson's videos and carry no severity labels** — they exercise
the pipeline only. Swap in your own clips by dropping them in `data/raw/`.


In [ ]:
import os
from remotezip import RemoteZip

URL  = 'https://ndownloader.figshare.com/files/35454242'
WANT = 'Videos/OAW01-bottom.mp4'
os.makedirs('data/raw', exist_ok=True)
out = 'data/raw/' + os.path.basename(WANT)

if not os.path.exists(out):
    with RemoteZip(URL) as z:
        with z.open(WANT) as src, open(out, 'wb') as dst:
            while chunk := src.read(1 << 20):
                dst.write(chunk)
print(out, round(os.path.getsize(out)/1e6, 1), 'MB')


## 4. Extract keypoints

`--seconds 25` keeps the demo quick. Drop the flag to process the full 75 s clip
(a few minutes on a Colab CPU).

The subject walks a long hall and is often only ~100 px tall in a 1080x1920 frame,
which is below what BlazePose's person detector fires on — raw full-frame detection
lands around 47%. The pipeline recovers ~100% by locating the walker via a static
median-background model, then cropping and upscaling around them before inference.


In [ ]:
!python src/extract_pose.py data/raw/OAW01-bottom.mp4 --model full --seconds 25


## 5. Watch it

Two renders: the true full frame, and a crop that follows the subject so the
skeleton is actually legible. The follow-cam reuses the saved keypoints, so it
costs no extra inference.

OpenCV writes `mp4v`, which Colab's HTML5 player cannot decode — so this
transcodes an excerpt to H.264 before embedding it.


In [ ]:
!python src/render_followcam.py data/raw/OAW01-bottom.mp4


In [ ]:
import subprocess, base64
from IPython.display import HTML, display

def show(path, start=8, dur=10, width=320):
    """Transcode an excerpt to H.264 and embed it inline."""
    out = '/tmp/preview.mp4'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',str(start),
                    '-i',path,'-t',str(dur),'-vcodec','libx264',
                    '-pix_fmt','yuv420p','-crf','24',out], check=True)
    b64 = base64.b64encode(open(out,'rb').read()).decode()
    display(HTML(f'<video width={width} controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show('outputs/annotated/OAW01-bottom_followcam.mp4')


In [ ]:
show('outputs/annotated/OAW01-bottom_pose.mp4', width=200)  # true full frame


## 6. Gait signals

Three figures: per-joint tracking quality, gait signal + cadence, and the six
masking conditions drawn on a real tracked skeleton.

Gait metrics use the **world landmarks** (`wx,wy,wz` — metric, hip-centred), not
image `x,y`. The subject walks toward and away from the camera, so pixel scale
drifts and the perspective envelope swamps the step cycle.


In [ ]:
!python src/gait_report.py outputs/keypoints/OAW01-bottom.parquet


In [ ]:
from IPython.display import Image, display
for name in ['quality','gait','masking_conditions']:
    display(Image(f'outputs/figures/OAW01-bottom_{name}.png', width=1000))


## 7. The keypoint table

One row per (frame, joint). This is what feeds the masking experiments.


In [ ]:
import pandas as pd
df = pd.read_parquet('outputs/keypoints/OAW01-bottom.parquet')
print(df.shape, '=', df.frame.nunique(), 'frames x', df.joint_id.nunique(), 'joints')
df.head()


## 8. Masking conditions

Indices each condition **hides during pretraining**; all 33 stay visible at
inference. Grounded in MDS-UPDRS Item 3.10 (stride amplitude, stride speed,
foot-lift height, heel strike, turning, arm swing).


In [ ]:
sys.path.insert(0, 'src')
from pose_topology import MASK_GROUPS, LANDMARK_NAMES
for name, idxs in MASK_GROUPS.items():
    joints = ', '.join(LANDMARK_NAMES[i] for i in idxs[:5])
    print(f'{name:14s} masks {len(idxs):2d}/33  ({joints}, ...)')


## Notes

- **Using your own video:** upload to `data/raw/`, then run cells 4-6 with your
  filename. The background model assumes a **static camera**; for handheld footage
  pass `--no-roi` and expect a lower detection rate.
- **Tracking quality varies by joint.** On this clip: torso 1.00, face 0.99,
  legs 0.89, arms 0.83 — with fingers (0.67-0.73) and heels (0.75) weakest.
  The arm landmarks that Model 3 masks are the least reliable ones, and they
  degrade further as the subject walks away from the camera.
- **Faces are blurred in this dataset**, so face landmarks drift. If the clinical
  corpus is likewise de-identified, the Model 6 face control masks keypoints that
  were never trustworthy — worth confirming before relying on it as a control.
- **Persisting outputs:** Colab wipes local files when the runtime ends. Mount
  Drive (`from google.colab import drive; drive.mount('/content/drive')`) and copy
  `outputs/` there, or commit figures back to the repo.
